# General joint phase retrieval
Minimal use of `phase_retrieval_core_general.py`.

In [ ]:
import numpy as np
from library import phase_retrieval_core_general as pr

In [ ]:
holograms = np.load("data/general_holograms.npy")  # (n_observations, nx, ny)
mask_pixel = np.load("data/mask_pixel.npy")
supportmask = np.load("data/supportmask.npy")
state_labels = ["saturated", "saturated", "domains", "domains"]
energy_labels = ["E1", "E1", "E1", "E1"]
polarizations = [+1, -1, +1, -1]
beam_labels = ["beam1", "beam1", "beam1", "beam1"]
saturated_states = {"saturated": +1}

In [ ]:
recipe = {
    # Independent update schedule for each observation.
    "inner_mode": ["HAPRE", "ER"],       # Algorithms repeated in every outer cycle.
    "inner_Nit": [700, 50],               # Iterations in each stage.
    "outer_iterations": 100,              # Number of update-plus-model-projection cycles.
    "warmup_mode": ["HAPRE"],            # Independent pre-coupling algorithms.
    "warmup_Nit": 0,                      # Zero disables warmup.
    "shuffle_observations": True,         # Randomize observation order.
    "random_seed": None,                  # Seed for update-order randomization.
    "beta_zero": 0.5,                     # Beta value(s), scalar or one per stage.
    "beta_mode": "arctan",               # Beta schedule name(s) or arrays.
    "alpha_zero": 0.0,                    # TV strength; zero disables TV.
    "alpha_mode": "const",               # Alpha schedule name(s) or arrays.
    "TV_freq": 1e9,                       # TV update interval.
    "warmup_beta_zero": None,             # None inherits the inner setting.
    "warmup_beta_mode": None,             # None inherits the inner setting.
    "warmup_alpha_zero": None,            # None inherits the inner setting.
    "warmup_alpha_mode": None,            # None inherits the inner setting.
    "warmup_TV_freq": None,               # None inherits the inner setting.
    "plot_every": 1e9,                    # Error sampling/plot interval.
    "average_img": 1,                     # Number of best late iterates to average.
    "Fourier_last": True,                 # Finish each stage with its Fourier constraint.
    "final_fourier_constraint": True,     # Finish outputs on measured amplitudes.
    "hologram_intensity_cutoff_vmin": -1, # Percentile baseline subtraction.
    # General metadata-aware model.
    "projection_model": "physical_factorized", # physical_factorized, state_energy_beam, or none.
    "projection_every": 1,                # Joint-projection interval.
    "projection_start": 0,                # First cycle eligible for projection.
    "projection_relaxation": 1.0,         # Projection blending fraction.
    "projection_constraints_inside_support_only": False, # If True, apply joint projections only inside supportmask.
    "observation_weights": None,          # Positive weight per observation.
    "rank_deficient": "error",           # Error or accept a minimum-norm linear solution.
    # Physical factorization L = C_beam + q_charge(E) + p*q_magnetic(E)*mz_state.
    "physical_iterations": 20,            # Alternating-fit iterations per projection.
    "saturated_states": saturated_states, # Optional state-to-+1/-1 mapping.
    "charge_spectral_constraint": "free",   # free, kk, known_beta, or known_beta_kk.
    "magnetic_spectral_constraint": "free", # Independent magnetic spectral constraint.
    "energy_values": None,                # Strictly increasing energies for KK constraints.
    "known_charge_beta_spectrum": None,   # Known charge absorption-like spectrum.
    "known_charge_delta_spectrum": None,  # Known charge dispersion-like spectrum.
    "known_magnetic_beta_spectrum": None, # Known magnetic absorption-like spectrum.
    "known_magnetic_delta_spectrum": None,# Known magnetic dispersion-like spectrum.
    "charge_absorption_part": "real",    # Charge absorption location: real or imag response part.
    "magnetic_absorption_part": "real",  # Magnetic absorption location.
    "charge_response_real_range": None,   # Optional bounds on Re(q_charge).
    "charge_response_imag_range": None,   # Optional bounds on Im(q_charge).
    "magnetic_response_real_range": None, # Optional bounds on Re(q_magnetic).
    "magnetic_response_imag_range": None, # Optional bounds on Im(q_magnetic).
    "kk_sign": 1.0,                       # KK sign convention.
    "kk_subtract_baseline": True,         # Remove endpoint baseline before KK.
    "kk_normalize_input": False,          # Normalize absorption before KK.
    "known_spectrum_normalization": "none", # none, maxabs, l2, or std.
    "fit_known_spectrum_scale": True,     # Fit supplied-spectrum scale.
    "fit_known_spectrum_offset": True,    # Fit supplied-spectrum offset.
    "log_floor": 1e-12,                   # Magnitude floor before complex logarithm.
}

fields, fieldswarmup, components, bsmasks, errors = (
    pr.general_phase_retrieval_algorithm(
        holograms,
        mask_pixel,
        supportmask,
        state_labels=state_labels,
        energy_labels=energy_labels,
        polarization_coefficients=polarizations,
        beam_labels=beam_labels,
        saturated_states=saturated_states,
        general_recipe=recipe,
    )
)